# Fake Email Detection: Supervised Algorithms
Run cells in order. Each model trains separately on the included labelled email dataset and prints test accuracy. Labels: spam/phishing = 1, ham/legitimate = 0.
Dataset: `../data/emails.csv`. These starter examples demonstrate the pipeline; use a larger, representative dataset for meaningful real-world performance.

In [ ]:
from pathlib import Path
import pandas as pd
# Load the labelled dataset included with this project.
DATA_FILE = Path('../data/emails.csv')
df = pd.read_csv(DATA_FILE)
df = df[['text', 'label']].dropna()
df['text'] = df['text'].astype(str)
label_map = {'ham':0, 'legitimate':0, 'legit':0, 'safe':0, '0':0, 'spam':1, 'phishing':1, 'fake':1, 'fraud':1, 'suspicious':1, '1':1}
df['label'] = df['label'].astype(str).str.strip().str.lower().map(label_map)
df = df.dropna(subset=['label']); df['label'] = df['label'].astype(int)
print('Dataset file:', DATA_FILE.resolve())
print('Total labelled emails:', len(df))
display(df['label'].value_counts().rename(index={0:'Legitimate / ham (0)', 1:'Suspicious / spam (1)'}))
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
X_train_text, X_test_text, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label'])
# Fit vectorizer only on training data
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print('Train rows:', X_train.shape[0], '| Test rows:', X_test.shape[0])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
# Algorithm 1: Logistic Regression
logistic_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
print(f'Logistic Regression accuracy: {logistic_accuracy*100:.2f}%')
print(classification_report(y_test, logistic_predictions, target_names=['Legitimate','Suspicious'], zero_division=0))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
# Algorithm 2: Multinomial Naive Bayes
naive_bayes_model = MultinomialNB()
naive_bayes_model.fit(X_train, y_train)
naive_bayes_predictions = naive_bayes_model.predict(X_test)
naive_bayes_accuracy = accuracy_score(y_test, naive_bayes_predictions)
print(f'Multinomial Naive Bayes accuracy: {naive_bayes_accuracy*100:.2f}%')
print(classification_report(y_test, naive_bayes_predictions, target_names=['Legitimate','Suspicious'], zero_division=0))

In [ ]:
from sklearn.svm import LinearSVC
# Algorithm 3: Linear Support Vector Machine
svm_model = LinearSVC(class_weight='balanced', random_state=42)
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)
print(f'Linear SVM accuracy: {svm_accuracy*100:.2f}%')
print(classification_report(y_test, svm_predictions, target_names=['Legitimate','Suspicious'], zero_division=0))

In [ ]:
# Compare each algorithm's held-out accuracy
accuracy_table = pd.DataFrame({'Algorithm':['Logistic Regression','Multinomial Naive Bayes','Linear SVM'], 'Accuracy (%)':[logistic_accuracy*100, naive_bayes_accuracy*100, svm_accuracy*100]}).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
display(accuracy_table.style.format({'Accuracy (%)':'{:.2f}%'}))
print('Top score for this split:', accuracy_table.loc[0,'Algorithm'])
print('Demo data score is not a reliable estimate of real-world accuracy.')

In [ ]:
# Try one email using the highest-scoring model
models = {'Logistic Regression':logistic_model, 'Multinomial Naive Bayes':naive_bayes_model, 'Linear SVM':svm_model}
best_name = accuracy_table.loc[0,'Algorithm']; best_model = models[best_name]
sample = ['Urgent: verify your account password using this link immediately']
prediction = int(best_model.predict(vectorizer.transform(sample))[0])
print('Model:', best_name)
print('Result:', 'Suspicious / spam' if prediction == 1 else 'Likely legitimate')